## ENEM-LLM-RAG — Classificação de Questões por Habilidade

Projeto de classificação de questões do ENEM (2009–2024) segundo a Matriz de Referência oficial (120 habilidades, 4 áreas de conhecimento), usando um LLM local (Ollama, gemma3:4b) com saída estruturada via Pydantic.

O objetivo é ir além do rótulo de disciplina (já disponível nos metadados) e atribuir a habilidade cognitiva específica exigida por cada questão — algo que não existe pronto nas bases públicas e que exige uma etapa de classificação semântica sobre o enunciado e a resposta correta.

**Roteiro do notebook:**
1. Análise exploratória dos dados consolidados
2. Organização e limpeza do dataset (múltiplas fontes, 2009–2024)
3. Baseline de classificação clássica (TF-IDF + Regressão Logística) como teste de viabilidade
4. Pipeline de classificação via LLM local, com schema estruturado, checkpointing e validação contra a matriz oficial

### Gráficos exploratórios e alguns resultados

Antes de qualquer modelagem, o objetivo aqui é entender três coisas: (1) o balanceamento das classes (disciplina, alternativa correta, distribuição por ano), (2) se o vocabulário das questões varia o suficiente entre disciplinas para justificar uma abordagem baseada em texto, e (3) problemas estruturais que podem limitar a classificação — em especial questões que dependem de figuras (não interpretáveis por um pipeline text-only) ou que carregam referências bibliográficas longas dentro do enunciado.

In [ ]:
import matplotlib.pyplot as plt

ax = df['correctAlternative'].value_counts().sort_index().plot.bar(color="black")

ax.set_title('Distribuição das respostas corretas')
ax.set_xlabel('Alternativa')
ax.set_ylabel('Quantidade')

plt.xticks(range(5), ['A', 'B', 'C', 'D', 'E'])
plt.show()

In [ ]:

ax = df['correctAlternative'].value_counts().sort_index().plot.bar(color="black")

df['discipline'].value_counts().plot.bar(color='black')
ax.set_title('Quantidade de questões de cada Disciplina')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

contagem = df.groupby(["year","discipline"]).size().unstack(fill_value=0)


plt.figure(figsize=(8,7))

sns.heatmap(
    contagem,
    annot=True,
    cmap="Blues",
    fmt="d",
    linewidths=.5
)

plt.title("Quantidade de questões por disciplina e ano após remoção de questões de língua extrangeira")
plt.xlabel("Disciplina")
plt.ylabel("Ano")
plt.show()

### Construção de uma lista de stopwords específicas para questões do ENEM

Além das remoções comuns de stopwords, para criar a WordCloud foram removidas algumas palavras que são comumente encontradas para referencias e guiar os alunos.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.corpus import stopwords

# Stopwords em português
stop_words = set(stopwords.words("portuguese"))

stop_words.update({
    "adaptado",
    "adaptada",
    "acesso",
    "disponível",
    "disponivel",
    "fonte",
    "acessado",
    "acessada",
    "www",
    "http",
    "https",
    "com",
    "br",
    "org",
    "gov",
    "ano",
    "figura",
    "imagem",
    "texto",
    "texto i",
    "texto ii",
    "cada",
    "desse",
    "dessa"
})

texto = " ".join(df["context"].dropna())

wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    stopwords=stop_words,
    collocations=False,
).generate(texto)

plt.figure(figsize=(14,7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud das Questões do ENEM")
plt.show()

In [ ]:
disciplinas = df["discipline"].unique()

for disc in disciplinas:
    texto = " ".join(
        df.loc[df["discipline"] == disc, "context"].dropna()
    )

    wc = WordCloud(
        width=1000,
        height=500,
        background_color="white",
        stopwords=stop_words,
        collocations=False,
    ).generate(texto)

    plt.figure(figsize=(12,5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(disc)
    plt.savefig(f"nuvem_{disc}.png")
    plt.show()

In [ ]:
import re
from collections import Counter
from nltk.corpus import stopwords


texto = " ".join(df["context"].dropna()).lower()

# Remove URLs
texto = re.sub(r"http\S+|www\.\S+", " ", texto)

# Remove pontuação
texto = re.sub(r"[^\w\s]", " ", texto)

# Remove números
texto = re.sub(r"\d+", " ", texto)

palavras = [
    p for p in texto.split()
    if p not in stop_words and len(p) > 2
]

frequencia = Counter(palavras)

top20 = frequencia.most_common(20)


top20 = pd.DataFrame(top20, columns=["Palavra", "Frequência"])

top20.plot.barh(
    x="Palavra",
    y="Frequência",
    figsize=(8,6),
    legend=False, 
    color = 'black'
)

plt.title("20 palavras mais frequentes")
plt.xlabel("Frequência")
plt.ylabel("")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

tfidf = model.named_steps["tfidf"]

X_tfidf = tfidf.transform(df["context"])

feature_names = tfidf.get_feature_names_out()

media = X_tfidf.mean(axis=0).A1

top20 = (
    pd.DataFrame({
        "Termo": feature_names,
        "TF-IDF": media
    })
    .sort_values("TF-IDF", ascending=False)
    .head(20)
)


plt.figure(figsize=(8,6))

plt.barh(top20["Termo"], top20["TF-IDF"], color='black')

plt.gca().invert_yaxis()

plt.title("20 termos com maior TF-IDF médio")
plt.xlabel("TF-IDF médio")

plt.tight_layout()
plt.show()

In [ ]:
# Criar colunas de análise
df['tem_figura'] = df['context'].str.contains(r'!\[.*?\]\(.*?\)', regex=True, na=False)
df['tem_referencia'] = df['context'].str.contains(r'[A-ZÀ-ÿ]+, [A-ZÀ-ÿ]\.', regex=True, na=False)

# Contagens
com_figura = df['tem_figura'].sum()
sem_figura = len(df) - com_figura
com_referencia = df['tem_referencia'].sum()

# Gráfico
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Figuras
axes[0].bar(['Sem figura', 'Com figura'], [sem_figura, com_figura], 
            color=['lightgray', 'steelblue'], edgecolor='black')
axes[0].set_title(f'Questões com Figuras\nTotal: {len(df)}')
for i, v in enumerate([sem_figura, com_figura]):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=11)

# Referências
axes[1].bar(['Sem referência', 'Com referência'], 
            [len(df) - com_referencia, com_referencia], 
            color=['lightgray', 'seagreen'], edgecolor='black')
axes[1].set_title(f'Questões com Referências')
for i, v in enumerate([len(df) - com_referencia, com_referencia]):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f"📊 Resumo:")
print(f"  Total de questões: {len(df)}")
print(f"  Com figuras: {com_figura} ({100*com_figura/len(df):.1f}%)")
print(f"  Com referências: {com_referencia} ({100*com_referencia/len(df):.1f}%)")

### Organizando o DataFrame de trabalho

As questões vêm de fontes públicas diferentes cobrindo períodos distintos (2009–2023 e 2024 separadamente), com pequenas diferenças de schema. Em particular, o campo `alternativesIntroduction` (frase que antecede as alternativas) só existe em parte dos dados — por isso é incorporado ao `context` antes do merge, para não perder essa informação nas questões que a possuem.

In [ ]:
import json

# Carregar 2009-2023
with open('questoes_enem_todas.json', 'r', encoding='utf-8') as f:
    questoes_2009_2023 = json.load(f)

# Carregar 2024
with open('2024.json', 'r', encoding='utf-8') as f:
    questoes_2024 = json.load(f)

# Juntar
todas = questoes_2009_2023 + questoes_2024

# Salvar
with open('enem_completo.json', 'w', encoding='utf-8') as f:
    json.dump(todas, f, ensure_ascii=False, indent=2)

print(f"2009-2023: {len(questoes_2009_2023)} questões")
print(f"2024: {len(questoes_2024)} questões")
print(f"✅ Total: {len(todas)} questões em enem_completo.json")

In [ ]:
import json

# Carregar 2009-2023
with open('questoes_enem_todas.json', 'r', encoding='utf-8') as f:
    questoes_2009_2023 = json.load(f)

# Juntar context + alternativesIntroduction nas anteriores|
for q in questoes_2009_2023:
    intro = q.get('alternativesIntroduction')
    ctx = q.get('context') or ''
    
    if intro:
        q['context'] = ctx + '\n' + intro
    q['alternativesIntroduction'] = None

# Carregar 2024
with open('2024.json', 'r', encoding='utf-8') as f:
    questoes_2024 = json.load(f)

# Juntar e salvar
todas = questoes_2009_2023 + questoes_2024

with open('enem_completo.json', 'w', encoding='utf-8') as f:
    json.dump(todas, f, ensure_ascii=False, indent=2)

print(f"✅ {len(todas)} questões salvas em enem_completo.json")

==========================================================================================================

### Importação E treinamento de Modelo

### Atribuição de disciplina por posição

O ENEM não fornece a disciplina de cada questão diretamente — ela é inferida pela posição (`index`) dentro da prova, que segue uma ordem fixa por área. Essa ordem muda a partir da reforma de 2017 (mudança na sequência das áreas), por isso a regra abaixo é condicional ao ano. Questões de língua estrangeira (inglês/espanhol) são removidas separadamente por terem habilidades próprias, fora do escopo tratado aqui.

In [ ]:
import pandas as pd

df = pd.read_json('enem_completo.json')

def atribuir_disciplina(index, year):

    if year <= 2016:
        if 1 <= index <= 45:
            return "ciencias-humanas"
        elif 46 <= index <= 90:
            return "ciencias-natureza"
        elif 91 <= index <= 135:
            return "linguagens"
        elif 136 <= index <= 180:
            return "matematica"

    else:
        if 1 <= index <= 45:
            return "linguagens"
        elif 46 <= index <= 90:
            return "ciencias-humanas"
        elif 91 <= index <= 135:
            return "ciencias-natureza"
        elif 136 <= index <= 180:
            return "matematica"

    return None


df["discipline"] = df.apply(
    lambda row: atribuir_disciplina(
        row["index"],
        row["year"]
    ),
    axis=1
)

print("Questões sem disciplina:",
      df["discipline"].isna().sum())

df.to_json(
    "enem_completo.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"✅ Disciplinas atribuídas. Total: {len(df)} questões")

A partir daqui os dados estão limpos e organizados. A partir daqui serão aplicados os modelos de ML necessários.

Questões → Embeddings → Clusters → Rotular 1 questão por cluster → Propagar rótulos

In [ ]:
def eh_lingua_estrangeira(index, year):

    if year <= 2016:
        # Linguagens está em 91–135
        return 131 <= index <= 135

    else:
        # Linguagens está em 1–45
        return 41 <= index <= 45

mask = (
    ((df["year"] <= 2016) &
     (df["index"].between(91,95)))
    |
    ((df["year"] >= 2017) &
     (df["index"].between(1,5)))
)

df = df[~mask].copy()

In [ ]:
import pandas as pd

mr = pd.read_csv('matriz_de_referencia.csv')

In [ ]:
df[df['language'] == 'ingles']


### Limpeza para uso no pipeline de classificação

Alguns ajustes aqui existem especificamente porque o texto será consumido por um LLM, não só analisado estatisticamente:

- A letra da alternativa correta (A–E) é trocada pelo **texto completo** da alternativa — uma letra isolada não carrega sinal semântico nenhum para o modelo.
- Links de imagem no `context` são substituídos pelo marcador `[FIGURA]`, documentando que a questão depende de conteúdo visual que o pipeline não processa (em vez de deixar um link quebrado no meio do enunciado).
- Questões anuladas, sem resposta, ou com resposta-placeholder são descartadas — não têm um rótulo verdadeiro contra o qual classificar.

In [ ]:
mapeamento = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
df['correctIndex'] = df['correctAlternative'].map(mapeamento)

def extrair_correta(alternativas):
    for alt in alternativas:
        if alt and alt.get('isCorrect'):
            return alt['text']
    return None

df['resposta_correta'] = df['alternatives'].apply(extrair_correta)

# Substituir links de imagens por [FIGURA]
df['context'] = df['context'].str.replace(
    r'!\[.*?\]\(https?://[^\s)]+\)', 
    '[FIGURA]', 
    regex=True
)


# Remover quebras de linha desnecessárias
df["context"] = df["context"].str.replace("\n", " ").str.replace("  ", " ")

    
# Lista de valores que você quer remover
valores_remover = ["Anulado", "Nulo", "Sem resposta", ""]

# Remove todos
df = df[~df["correctAlternative"].isin(valores_remover)]

In [ ]:
df.drop(columns=['files','alternativesIntroduction', 'alternatives', 'correctAlternative', 'language', 'year', 'index','title'], inplace=True)

In [ ]:
# Ver quantas têm placeholder
placeholders = df['resposta_correta'].str.contains('\[\[placeholder\]\]', regex=True, na=False)
print(f"Questões com placeholder: {placeholders.sum()}")

# Remover
df = df[~placeholders].copy()

### Teste dos modelos

**1. TF-IDF + Regressão Logística (baseline)**

Antes de investir num pipeline de LLM para o problema real (~120 habilidades, poucos exemplos por classe), faz sentido testar um baseline leve e 100% local num problema mais fácil e com groundtruth confiável: prever `discipline` (só 4 classes, já conhecida via metadados). Se um modelo clássico já tiver dificuldade de generalizar nesse caso simples, é um sinal forte de que o problema de 120 classes — com amostras ainda menores e mais ruidosas por classe — não é resolvível de forma supervisionada clássica.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#===========================================================

from sklearn.metrics import classification_report


In [ ]:
df["text"] = (
    "Questão: " + df["context"].astype(str).str.strip()
    + "\nResposta correta: " + df["resposta_correta"].astype(str).str.strip()
)

y = df['discipline']
X = df['text']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
from sklearn.model_selection import GridSearchCV

# Modelo base
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(
        penalty="l2",
        solver='lbfgs',  # ← TROCAR para lbfgs
        max_iter=2000,
        random_state=42
    ))
])

# Grid com C ≤ 1 e foco em reduzir overfitting
param_grid = {
    # TF-IDF
    "tfidf__ngram_range": [(1,2), (1,3)],  # (1,1) é muito simples, (1,5) overfit
    "tfidf__min_df": [2, 3, 5],            # Remove palavras raras
    "tfidf__max_df": [0.85, 0.90, 0.92],   # Remove palavras muito comuns
    "tfidf__max_features": [2000, 3000, 4000, 5000],  # CRUCIAL!
    "tfidf__sublinear_tf": [True],          # Fixo (melhor para generalização)
    "tfidf__use_idf": [True, False],        # Testar com/sem IDF
    
    # Classificador (C ≤ 1)
    "clf__C": [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 1.0],  # GRANULAR!
    "clf__class_weight": [None, 'balanced']  # Ajuda com classes desbalanceadas
    
}

# GridSearch com retorno de treino
grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
    verbose=1  # Mostra progresso
)

print("Iniciando GridSearch...")
grid.fit(X_train, y_train)

# Resultados
print(f"\n✅ Melhor modelo encontrado:")
print(f"Params: {grid.best_params_}")
print(f"CV Score: {grid.best_score_:.3f}")

In [ ]:
# ============================================
# AVALIAÇÃO DO MELHOR MODELO
# ============================================

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Melhor modelo


# Previsões
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Métricas
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print("="*60)
print("📊 RESULTADOS FINAIS")
print("="*60)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Gap:            {train_acc - test_acc:.4f}")

print("\n📋 CLASSIFICATION REPORT (Teste):")
print(classification_report(y_test, y_pred_test))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_test)
classes = best_model.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusão - Teste')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

# Análise por classe
print("\n📈 PERFORMANCE POR CLASSE:")
print("-" * 50)
f1_scores = classification_report(y_test, y_pred_test, output_dict=True)

for classe in classes:
    f1 = f1_scores[classe]['f1-score']
    precision = f1_scores[classe]['precision']
    recall = f1_scores[classe]['recall']
    support = f1_scores[classe]['support']
    
    print(f"{classe:20s} | F1: {f1:.3f} | P: {precision:.3f} | R: {recall:.3f} | N: {support}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score


best_model = grid.best_estimator_
# ==========================================================
# MODEL EVALUATION
# ==========================================================

# Predictions
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
gap = train_acc - test_acc

print("="*70)
print("MODEL EVALUATION")
print("="*70)

summary = pd.DataFrame({
    "Metric": [
        "Train Accuracy",
        "Test Accuracy",
        "Overfitting Gap"
    ],
    "Value": [
        train_acc,
        test_acc,
        gap
    ]
})

display(summary)

# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_test))

# ==========================================================
# CONFUSION MATRIX
# ==========================================================

fig, ax = plt.subplots(figsize=(8,6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_test,
    cmap="Blues",
    xticks_rotation=45,
    ax=ax
)

plt.title("Confusion Matrix")
plt.show()

# ==========================================================
# F1 SCORE POR CLASSE
# ==========================================================

report = classification_report(
    y_test,
    y_pred_test,
    output_dict=True
)

classes = best_model.classes_

f1 = [
    report[c]["f1-score"]
    for c in classes
]

plt.figure(figsize=(8,4))

plt.bar(classes, f1, color='black')

plt.ylabel("F1-score")
plt.ylim(0,1)

plt.title("F1-score by Class")

plt.xticks(rotation=20)

plt.show()

# ==========================================================
# CROSS VALIDATION
# ==========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("\nCross Validation")
print(f"Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ==========================================================
# COMPARAÇÃO HOLDOUT x CROSS VALIDATION
# ==========================================================

comparison = pd.DataFrame({
    "Evaluation":[
        "Train",
        "Test",
        "Cross Validation"
    ],
    "Accuracy":[
        train_acc,
        test_acc,
        cv_scores.mean()
    ]
})

display(comparison)

plt.figure(figsize=(6,4))

bars = plt.bar(
    comparison["Evaluation"],
    comparison["Accuracy"],
    color="black"
)

plt.ylim(0, 1.02)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.01,
        f"{bar.get_height():.3f}",
        ha="center",
        fontsize=10
    )

plt.ylabel("Accuracy")
plt.title("Model Performance")

plt.tight_layout()
plt.show()
# ==========================================================
# GRID SEARCH RESULT
# ==========================================================

print("\nBest Parameters")
print(grid.best_params_)

print(f"\nBest CV Score: {grid.best_score_:.4f}")

# ==========================================================
# CONCLUSION
# ==========================================================

print("\nConclusion")

print(f"• Holdout Accuracy : {test_acc:.4f}")
print(f"• Cross-validation : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"• Overfitting Gap  : {gap:.4f}")

if gap < 0.08:
    print("\nModel presents good generalization.")
elif gap < 0.12:
    print("\nModel presents moderate overfitting.")
else:
    print("\nModel presents significant overfitting.")

### Leitura do resultado

Mesmo após tuning de hiperparâmetros via GridSearchCV, o melhor modelo manteve um gap perceptível entre treino e teste/validação cruzada — isso na tarefa mais fácil (4 classes balanceadas o suficiente). Esse resultado reforça a decisão de não tentar classificação supervisionada clássica diretamente nas 120 habilidades, e motiva a mudança para uma abordagem *zero-shot* baseada na descrição oficial de cada habilidade, na próxima seção.

### Pipeline de classificação via LLM local

A partir daqui, a integração passa a ser feita com um modelo Ollama local, aplicada inicialmente a apenas uma parte dos dados por limitações de hardware. Principais decisões de design:

- **Execução local (Ollama, gemma3:4b)** — sem custo de API e sem enviar dados para fora, ao custo de rodar sobre uma amostra do dataset neste notebook, e não o conjunto completo.
- **Saída estruturada via schema Pydantic** (`HabilidadeClassificacao`), passada como `format` para o Ollama — garante JSON parseável em vez de texto livre, eliminando parsing frágil por regex.
- **Restrição explícita à Matriz de Referência da área** no prompt, para evitar que o modelo invente códigos de habilidade inexistentes.
- **Campo `habilidade_secundaria` opcional** — evita forçar uma decisão artificial de rótulo único quando duas habilidades são genuinamente centrais (raro, mas acontece).
- **`confianca` e `justificativa`** tornam a classificação auditável, não uma caixa-preta — importante para revisão manual posterior.
- **Validação pós-hoc (`resolver`)**: qualquer código retornado que não exista na matriz da área vira um `alerta` para revisão, em vez de ser aceito silenciosamente.
- **Checkpointing** a cada `CHECKPOINT_EVERY` questões e **retry com limite** tornam o processamento resiliente a falhas em execuções longas de LLM local.
- Processamento **sequencial por design**: a maioria das instalações locais do Ollama atende uma geração por vez, então paralelizar sem `OLLAMA_NUM_PARALLEL` configurado só competiria por VRAM sem ganho real.

In [ ]:
df = df.reset_index()


In [ ]:
import os
import pandas as pd
import ollama
from pydantic import BaseModel, Field, ValidationError
from tqdm import tqdm


In [ ]:
MODEL = "gemma3:4b"        # troque pelo modelo que você tem puxado (`ollama list`)
NUM_CTX = 8192                # contexto — Ollama trunca silenciosamente se ficar baixo demais
CHECKPOINT_PATH = "checkpoint_habilidades.csv"
CHECKPOINT_EVERY = 25

COL_ID = "index"
COL_CONTEXT = "context"          # enunciado / contexto da questão
COL_RESPOSTA = "resposta_correta"
COL_TEXT = "text"                # coluna combinada, gerada por preparar_texto()
COL_DISCIPLINA = "discipline"

DISCIPLINA_TO_AREA = {
    "linguagens": "Linguagens, Códigos e suas Tecnologias",
    "matematica": "Matemática e suas Tecnologias",
    "ciencias-humanas": "Ciências Humanas e suas Tecnologias",
    "ciencias-natureza": "Ciências da Natureza e suas Tecnologias",
}

class HabilidadeClassificacao(BaseModel):
    raciocinio: str = Field(description="Comparação breve entre as habilidades candidatas, indicando se mais de uma é genuinamente central para resolver a questão")
    habilidade_principal: str = Field(description="Código da habilidade principal, ex: H05")
    habilidade_secundaria: str = Field(description="Código de uma segunda habilidade IGUALMENTE central pra resolver a questão. Use string vazia \"\" se não houver — deve ser a maioria dos casos")
    confianca: float = Field(ge=0.0, le=1.0, description="Confiança da classificação, de 0.0 a 1.0")
    justificativa: str = Field(description="Justificativa em no máximo 3 frases, cobrindo a habilidade principal e, se houver, a secundária")

PROMPT_TEMPLATE = """Você é um especialista na elaboração e análise de itens do Exame Nacional do Ensino Médio (ENEM).

Sua tarefa é identificar qual habilidade da Matriz de Referência do ENEM é a mais adequada para a questão apresentada.

Utilize EXCLUSIVAMENTE as habilidades presentes na Matriz de Referência fornecida abaixo. Nunca invente códigos ou 
habilidades que não estejam na matriz.

Critérios para a classificação:

1. Leia atentamente o enunciado e a resposta correta.
2. Identifique a(s) principal(is) competência(s) cognitiva(s) exigida(s) para resolver a questão.
3. Compare essas competências com TODAS as habilidades da Matriz de Referência.
4. Escolha a habilidade PRINCIPAL: aquela que melhor representa o objetivo central da questão.
5. Só preencha uma habilidade SECUNDÁRIA se uma segunda habilidade for GENUINAMENTE tão central quanto a principal — não inclua habilidades apenas relacionadas ao tema. Na maioria das questões não deve haver secundária.
6. Se duas ou mais habilidades parecerem adequadas como principal, escolha aquela cuja descrição seja mais específica.

IMPORTANTE:

- Não classifique apenas pelo tema ou assunto da questão.
- Baseie sua decisão na habilidade cognitiva exigida para resolver corretamente a questão.
- Os códigos devem ser exatamente os presentes na matriz (por exemplo: H05). Nunca invente um código inexistente.
- Se não houver habilidade secundária, deixe o campo "habilidade_secundaria" como string vazia "".
- Analise cada questão de forma independente. Não tente distribuir as respostas entre as habilidades.

## Matriz de Referência ({area})

{matriz}

## Questão

{texto}
"""

In [ ]:
def build_matriz_text(df_matriz: pd.DataFrame, area: str) -> str:
    subset = df_matriz[df_matriz["area"] == area]
    if subset.empty:
        raise ValueError(f"Nenhuma habilidade encontrada para area='{area}'. Verifique DISCIPLINA_TO_AREA.")
    return "\n".join(f"{r.habilidade_codigo}: {r.habilidade_descricao}" for r in subset.itertuples())


def preparar_texto(df: pd.DataFrame) -> pd.DataFrame:
    """Combina enunciado + resposta correta em um único bloco rotulado (coluna COL_TEXT).

    Mantém rastreio de linhas sem resposta preenchida em "sem_resposta" — isso é
    sinal de qualidade de dado que vale revisar à parte, em vez de deixar sumir
    silenciosamente dentro do texto combinado.
    """
    df = df.copy()
    df["sem_resposta"] = df[COL_RESPOSTA].isna() | (df[COL_RESPOSTA].astype(str).str.strip() == "")

    media_len = df[COL_RESPOSTA].astype(str).str.len().mean()
    if media_len < 5:
        print(
            f"[aviso] resposta_correta tem média de {media_len:.1f} caracteres — "
            f"pode ser só a letra (A-E) em vez do texto da alternativa. "
            f"Se for o caso, faça o join com o texto da alternativa antes de continuar."
        )

    df[COL_TEXT] = (
        "Questão: " + df[COL_CONTEXT].astype(str).str.strip()
        + "\nResposta correta: " + df[COL_RESPOSTA].astype(str).str.strip()
    )
    return df


def build_prompt(row: pd.Series, df_matriz: pd.DataFrame) -> str:
    area = DISCIPLINA_TO_AREA[row[COL_DISCIPLINA]]
    matriz_text = build_matriz_text(df_matriz, area)
    return PROMPT_TEMPLATE.format(
        area=area,
        matriz=matriz_text,
        texto=row[COL_TEXT],
    )


def classificar_questao(row: pd.Series, df_matriz: pd.DataFrame, max_retries: int = 3) -> dict:
    try:
        area = DISCIPLINA_TO_AREA[row[COL_DISCIPLINA]]
        matriz_area = df_matriz[df_matriz["area"] == area]
        prompt = build_prompt(row, df_matriz)
    except Exception as e:
        return {COL_ID: row[COL_ID], "erro": f"falha ao montar prompt: {e}"}

    for attempt in range(max_retries):
        try:
            resp = ollama.chat(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                format=HabilidadeClassificacao.model_json_schema(),
                options={"temperature": 0, "num_ctx": NUM_CTX},
            )
            parsed = HabilidadeClassificacao.model_validate_json(resp["message"]["content"])
            result = parsed.model_dump()
            result[COL_ID] = row[COL_ID]

            def resolver(codigo: str):
                if not codigo:
                    return None, None
                match = matriz_area[matriz_area["habilidade_codigo"] == codigo]
                if match.empty:
                    return None, f"código '{codigo}' não existe na área '{area}'"
                return match["habilidade_descricao"].iloc[0], None

            desc_p, alerta_p = resolver(parsed.habilidade_principal)
            desc_s, alerta_s = resolver(parsed.habilidade_secundaria)
            result["descricao_principal"] = desc_p
            result["descricao_secundaria"] = desc_s

            alertas = [a for a in (alerta_p, alerta_s) if a]
            if alertas:
                result["alerta"] = "; ".join(alertas) + " — revisar manualmente"

            return result
        except Exception as e:
            if attempt == max_retries - 1:
                return {COL_ID: row[COL_ID], "erro": str(e)}
            continue
    return {COL_ID: row[COL_ID], "erro": "esgotou tentativas"}

def rodar_pipeline(df_questoes: pd.DataFrame, df_matriz: pd.DataFrame) -> pd.DataFrame:
    if COL_TEXT not in df_questoes.columns:
        df_questoes = preparar_texto(df_questoes)

    ja_processados = set()
    resultados = []
    if os.path.exists(CHECKPOINT_PATH):
        checkpoint = pd.read_csv(CHECKPOINT_PATH)
        resultados = checkpoint.to_dict("records")
        ja_processados = set(checkpoint[COL_ID])
        print(f"Retomando checkpoint: {len(ja_processados)} questões já processadas.")

    pendentes = df_questoes[~df_questoes[COL_ID].isin(ja_processados)]

    # Sequencial de propósito: a maioria das instalações locais do Ollama processa
    # uma requisição de geração por vez (mesmo que aceite a conexão em paralelo),
    # então threads aqui só enfileiram sem ganho — e podem competir por VRAM.
    # Se você configurou OLLAMA_NUM_PARALLEL > 1 no servidor, pode paralelizar
    # com ThreadPoolExecutor como no script da API, mas teste antes.
    for i, (_, row) in enumerate(tqdm(pendentes.iterrows(), total=len(pendentes)), start=1):
        resultados.append(classificar_questao(row, df_matriz))
        if i % CHECKPOINT_EVERY == 0:
            pd.DataFrame(resultados).to_csv(CHECKPOINT_PATH, index=False)

    df_resultado = pd.DataFrame(resultados)
    df_resultado.to_csv(CHECKPOINT_PATH, index=False)
    return df_resultado


### Validação em amostra antes da execução completa

Antes de rodar o pipeline completo (custoso em tempo, por ser local e sequencial), uma amostra de 20 questões é processada isoladamente para validar o pipeline de ponta a ponta — taxas de `erro` e `alerta` servem como indicador rápido de saúde do pipeline antes do commit de tempo na execução completa sobre o dataset consolidado.

In [ ]:

df_questoes = df.copy()   # ajuste o caminho
df_matriz = mr.copy()          # ajuste o caminho

df_resultado = rodar_pipeline(df_questoes, df_matriz)
df_final = df_questoes.merge(df_resultado, on=COL_ID, how="left")
df_final.to_parquet("questoes_enem_com_habilidades.parquet", index=False)

erros = df_final[df_final["erro"].notna()] if "erro" in df_final.columns else pd.DataFrame()
alertas = df_final[df_final["alerta"].notna()] if "alerta" in df_final.columns else pd.DataFrame()
print(f"Concluído. {len(df_final)} questões, {len(erros)} com erro, {len(alertas)} com código fora da matriz.")

In [ ]:
CHECKPOINT_PATH = "checkpoint_teste.csv"   # separa do checkpoint do lote completo

df_questoes = df.copy()
df_matriz = mr.copy()

amostra = df_questoes.sample(20, random_state=0)

df_resultado = rodar_pipeline(amostra, df_matriz)
df_final = amostra.merge(df_resultado, on=COL_ID, how="left")

erros = df_final[df_final["erro"].notna()] if "erro" in df_final.columns else pd.DataFrame()
alertas = df_final[df_final["alerta"].notna()] if "alerta" in df_final.columns else pd.DataFrame()
print(f"Concluído. {len(df_final)} questões, {len(erros)} com erro, {len(alertas)} com código fora da matriz.")

In [ ]:
df_final

### Conclusão e próximos passos

- Validar uma amostra dos rótulos gerados pelo LLM contra julgamento humano/especialista, para estimar acurácia real (não só taxa de erro técnico).
- Analisar a distribuição de `confianca` e a taxa de `alerta` para priorizar quais questões revisar manualmente primeiro.
- Tratar separadamente as questões marcadas com `[FIGURA]`, já que dependem de conteúdo que o pipeline text-only não classifica de forma confiável.
- Escalar a execução para o dataset completo (~2900 questões) uma vez validada a qualidade na amostra.